In [26]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

cuda


In [27]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']


In [28]:
string_to_int = {ch:i for i, ch in enumerate(chars) }
int_to_string = {i:ch for i, ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([80,  1, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1,
        47, 33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,
         0,  0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,
         0,  1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36,
        25, 38, 28,  1, 39, 30,  1, 39, 50,  9])


In [29]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x)
print('targets:')
print(y)



tensor([137992, 134474,  28034, 160627])
inputs:
tensor([[67,  1, 62, 67,  1, 61, 62, 72],
        [61, 54, 57,  1, 67, 68, 73,  1],
        [61, 62, 66,  1, 59, 54, 65, 65],
        [76, 72,  1, 61, 68, 76, 11,  1]], device='cuda:0')
targets:
tensor([[67,  1, 62, 67,  1, 61, 62, 72,  0],
        [61, 54, 57,  1, 67, 68, 73,  1, 73],
        [61, 62, 66,  1, 59, 54, 65, 65,  1],
        [76, 72,  1, 61, 68, 76, 11,  1, 26]], device='cuda:0')


In [30]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [34]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


a4Jn;E"C v;O45I'(mdHF﻿'qJH[Hc8R)sxACg&;baq]:[wsxXF﻿,fFJB:zjmJsY J[e.qd2BY3B﻿a2wa0kv)t-Wy-fJKp,jvgJ);[L26PakM1S0OwP5qtrma5IMGN3Mmnnd7PJ)NK,RYep,YKG?"y?fYIXQ7?n7y? kDKd0*2[e6VA6VS'vItPM
4JrwXJE(Xud5-l&;459BiB IAqQ1gpuv49﻿ s
 L32MKI45rHZe!y0yFn,;32UE?7_HFAU'8.oy0bp"F"e.KxO Tzj6UP[9n;),;sj]﻿**FCFhbdqD)Mg49BJ!)D82*?n!yNsjv?6rlgiuQ tc*VvDK]3*?d[sHkuc3c_vN7fYFKD.Yna_P&fSVkny?vGX,0u.hTOM;dHODSFeY[﻿6!br[H!yE9L0e_lTPhO7cu1-c3Bi"]C;E,_3(7A86Pt[]PmQN2a
jd.
ROFAIEv'aqon,Od4l5
]P0jCFlG:gsSg36'B9DAwqo0(gL:w&0



In [35]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

tensor([151958,  39214,  38085, 185225])


RuntimeError: shape '[32]' is invalid for input of size 36

In [65]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


NkXFGW * ﻿26eUa5[NnODHQAMN]﻿T*k"7GkPK]euybUX6NVrS?)_7Vw7Kk"33sVcctFnOF0vMo),elEjc11QLI﻿ZY0FTk]t5:GNU.]3"_!U7pqKStDny:o)BM_r_k,wjO(yKfmU:'3;&vH)QXLJU-R)_7AFsJ:G:qpT.j&Va4z:e
WJT*y2lZMgRYg0H9JohgGdQ5O
,k
WkDJoZLRSJi
Gb'X_JRB:rLRAmSuyCK]Vj!]'qpD_5Sw!xe8Oo&fI2?8w3dfmY O(w_5Sbh!Z:"C4oR.b_w:GNgw0WuaRBND'gS):lF4q3'Y&Me2G15krUKZ7G;oGNho
6bjtLfM_znFIjmfD-h*k[bkDuy70gMq_!")Qp﻿B)n'oK(_Mwxo,
WFmOr&CK].YwV0Ajh*4H'H'[l"ZFIfgRbhos3]
aawUE Lk*L-N]n_4ZqR0vEa6")_H zny?ABMd*4mn(ls
Wuma(!X&l-GK4mr&*﻿QuprB'7cW;(W-T'
